# MCS Track Statistics: IMERG vs SCREAM — Tropics

Compares MCS lifetime statistics between IMERG observations and SCREAM model output
for tropical MCS tracks.

---

### Preprocessing — run once per source (outside this notebook)

The heavy lifting (read netCDF, filter to region, flatten jagged arrays) is done by
`scripts/combine_filter_mcs_trackstats.py`.  It reads yearly `mcs_tracks_final_*.nc`
files, filters tracks to a lat/lon box, flattens each file to a tidy DataFrame (one
row per valid time step, clipped to `track_duration`), and writes a single compressed
Parquet file per source.

```bash
# SCREAM
python scripts/combine_filter_mcs_trackstats.py \
    --indir /pscratch/sd/f/feng045/SCREAM-decadal/ne1024/stats \
    --region tropics --start-year 1995 --end-year 2005

# IMERG
python scripts/combine_filter_mcs_trackstats.py \
    --indir /pscratch/sd/f/feng045/waccem/mcs_global_v3/MCSMIP/stats \
    --pattern 'mcsmip_mcs_tracks_final_*.nc' \
    --region tropics --start-year 1998 --end-year 2024
```

Re-run the script whenever the year range changes; re-run this notebook to update plots.

---

### Notebook steps

1. **Load** — read the precomputed Parquet files (seconds, not minutes)
2. **Land / Ocean classification** — split by lifetime-min/max `pf_landfrac`
3. **Per-track lifetime statistics** — aggregate to one row per track
4. **Grouped box plots** — x = Land/Ocean, hue = source (IMERG, SCREAM, ...)
5. **Composite time-evolution** — mean of land MCS by track-duration bin

In [ ]:
import glob
import os
import time
import warnings
import re
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import display

%matplotlib inline
sns.set_theme(style='whitegrid', font_scale=1.2)

## Configuration

In [ ]:
# ── Figure output directory ────────────────────────────────────────────────────
figdir_out = '/global/cfs/cdirs/m1867/zfeng/SCREAM-decadal/figures/'

# ── Source configuration ───────────────────────────────────────────────────────
# Each entry: label, combined_dir (where the preprocessed Parquet lives),
#             region, and per-source start/end year.
#
# The year range must match what was passed to combine_filter_mcs_trackstats.py
# because they are baked into the Parquet filename:
#   mcs_tracks_final_combined_{region}_{start_year}_{end_year}.parquet
#
#   IMERG  available on disk: 1998–2024
#   SCREAM available on disk: 1995–2005
REGION = 'tropics'

SOURCES = {
    'imerg': dict(
        label        = 'IMERG',
        combined_dir = '/pscratch/sd/f/feng045/waccem/mcs_global_v3/MCSMIP/stats',
        region       = REGION,
        start_year   = 2007,   # ← edit to change range (re-run script if changed)
        end_year     = 2017,
    ),
    'ne1024': dict(
        label        = 'SCREAM',
        combined_dir = '/pscratch/sd/f/feng045/SCREAM-decadal/ne1024/stats',
        region       = REGION,
        start_year   = 1995,   # ← edit to change range (re-run script if changed)
        end_year     = 2005,
    ),
}

SOURCE_LABELS = {k: v['label'] for k, v in SOURCES.items()}
SOURCE_KEYS   = list(SOURCES)   # ['imerg', 'ne1024'] — drives all run loops


---
## Step 1 — Load preprocessed combined Parquet files

Each source's combined Parquet was produced offline by
`scripts/combine_filter_mcs_trackstats.py` (see intro above).
The script handled file discovery, region filtering, and flattening the jagged
`(tracks × times)` NetCDF arrays — eliminating the slow `xr.concat(join='outer')`
that used to run inside the notebook.

`load_combined_sources()` reads the Parquet for each source, assigns a `source` label,
and concatenates them into a single tidy DataFrame `df` (one row per valid time step).

In [ ]:
def combined_path(cfg):
    """Build the expected path to the preprocessed Parquet for one source."""
    fname = (f"mcs_tracks_final_combined_{cfg['region']}"
             f"_{cfg['start_year']}_{cfg['end_year']}.parquet")
    return os.path.join(cfg['combined_dir'], fname)


def load_combined_sources():
    """
    Load all source Parquet files and combine into one tidy DataFrame.

    The Parquet files are source-agnostic (produced by combine_filter_mcs_trackstats.py).
    The 'source' label (e.g. 'IMERG', 'SCREAM') is assigned here after loading.

    Returns
    -------
    pd.DataFrame with one row per valid (track, time-step); columns include
    'local_track', 'relative_step', 'track_duration', 'track_duration_h',
    'period', 'source', and all physics variables.
    """
    frames = []
    for src, cfg in SOURCES.items():
        path = combined_path(cfg)
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{cfg['label']}: preprocessed Parquet not found:\n"
                f"  {path}\n\n"
                f"  Generate it with:\n"
                f"    python scripts/combine_filter_mcs_trackstats.py \\\n"
                f"        --indir {cfg['combined_dir']} \\\n"
                f"        --region {cfg['region']} \\\n"
                f"        --start-year {cfg['start_year']} "
                f"--end-year {cfg['end_year']}"
            )
        d = pd.read_parquet(path)
        d['source'] = cfg['label']   # assign source label after load
        frames.append(d)
        print(f"{cfg['label']:>15s}: {len(d):,} rows | "
              f"{d['local_track'].nunique():,} tracks | "
              f"{os.path.basename(path)}")
    return pd.concat(frames, ignore_index=True)


# ── Run ────────────────────────────────────────────────────────────────────────
df = load_combined_sources()
print(f'\nCombined DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(df['source'].value_counts().rename('rows').to_frame())
df.head()


---
## Quick summary statistics

In [ ]:
# One row per track (drop duplicate time steps)
df_tracks = df.drop_duplicates(subset=['source', 'local_track'])

print('Track count and duration summary per source:')
summary = (
    df_tracks.groupby('source')['track_duration_h']
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    [['count', 'mean', '25%', '50%', '75%', '95%', 'max']]
    .round(1)
)
display(summary)

# Period breakdown (tracks per source-year)
print('\nTracks per period per source:')
period_counts = (
    df_tracks.groupby(['period', 'source'])
    .size()
    .unstack(fill_value=0)
)
display(period_counts)


---
## Step 5 — Land / Ocean track classification

Filter tracks by the **lifetime min/max of `pf_landfrac`** (PF land fraction, 0–1):
- **Land**: lifetime-min `pf_landfrac > LAND_MINFRAC` (0.9) — PF is never more than 10% over water at any time step
- **Ocean**: lifetime-max `pf_landfrac < OCEAN_MAXFRAC` (0.1) — PF is never more than 10% over land at any time step

The filter is applied at the time-step level: `df_tracks_l` and `df_tracks_o` contain all time-step rows for the qualifying tracks.

In [ ]:
# ── Land / Ocean classification thresholds (user-adjustable) ─────────────────
LAND_MINFRAC  = 0.9   # lifetime-min pf_landfrac must exceed this → land track
OCEAN_MAXFRAC = 0.1   # lifetime-max pf_landfrac must be below this → ocean track


def classify_land_ocean(df, land_min=LAND_MINFRAC, ocean_max=OCEAN_MAXFRAC,
                         landfrac_col='pf_landfrac'):
    """
    Split a time-step level DataFrame into land and ocean MCS subsets.

    Uses per-track lifetime min/max of `landfrac_col` (NaN-safe):
      land  : lifetime-min landfrac > land_min   (PF mostly over land at all steps)
      ocean : lifetime-max landfrac < ocean_max  (PF mostly over water at all steps)

    Returns
    -------
    df_l : time-step rows for land tracks
    df_o : time-step rows for ocean tracks
    """
    lf = (
        df.groupby(['source', 'local_track'])[landfrac_col]
        .agg(lf_min='min', lf_max='max')
        .reset_index()
    )

    land_ids  = lf.loc[lf['lf_min'] > land_min,  ['source', 'local_track']]
    ocean_ids = lf.loc[lf['lf_max'] < ocean_max, ['source', 'local_track']]

    df_l = df.merge(land_ids,  on=['source', 'local_track'])
    df_o = df.merge(ocean_ids, on=['source', 'local_track'])
    return df_l, df_o


# ── Run ───────────────────────────────────────────────────────────────────────
df_tracks_l, df_tracks_o = classify_land_ocean(df)

# Tag each subset with a surface label (carried into Step 6 agg and Step 7 plot)
df_tracks_l['surface'] = 'Land'
df_tracks_o['surface'] = 'Ocean'

# Summary: unique tracks per source
print(f'Land / Ocean classification  '
      f'(land_min={LAND_MINFRAC}, ocean_max={OCEAN_MAXFRAC})')
print()

rows = []
for src in SOURCE_KEYS:
    src_label = SOURCE_LABELS[src]
    nt_total = df[df['source'] == src_label]['local_track'].nunique()
    nt_land  = df_tracks_l[df_tracks_l['source'] == src_label]['local_track'].nunique()
    nt_ocean = df_tracks_o[df_tracks_o['source'] == src_label]['local_track'].nunique()
    rows.append({'source': src_label, 'all_tracks': nt_total,
                 'land_tracks': nt_land, 'ocean_tracks': nt_ocean,
                 'land_%': f'{nt_land/nt_total:.1%}',
                 'ocean_%': f'{nt_ocean/nt_total:.1%}'})

display(pd.DataFrame(rows).set_index('source'))


---
## Step 6 — Per-track lifetime statistics

Aggregate the time-step level DataFrames to **one row per track** using heterogeneous aggregations (max, min, mean, sum).  Adapt the variable lists below to add or remove statistics as needed.

In [ ]:
# Derived variables (computed on both land and ocean time-step DataFrames)
# ccs_area_all    : total CCS area including merged and split cloud systems
# core_area_ratio : convective core fraction of the total CCS area
# heavy_rain_ratio: fraction of total rainfall that is heavy rain
for _df in (df_tracks_l, df_tracks_o):
    _df['ccs_area_all']     = _df['ccs_area'] + _df['merge_ccs_area'] + _df['split_ccs_area']
    _df['core_area_ratio']  = _df['core_area'] / _df['ccs_area']
    _df['heavy_rain_ratio'] = _df['total_heavyvolrain'] / _df['total_volrain']
    # Replace inf (division by zero when denominator == 0) with NaN
    _df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Derived columns added to land and ocean DataFrames:")
for col in ['ccs_area_all', 'core_area_ratio', 'heavy_rain_ratio']:
    print(f"  {col}:")
    print(f"    Land : min={df_tracks_l[col].min():.4g},  "
          f"max={df_tracks_l[col].max():.4g},  "
          f"NaN={df_tracks_l[col].isna().sum():,}")
    print(f"    Ocean: min={df_tracks_o[col].min():.4g},  "
          f"max={df_tracks_o[col].max():.4g},  "
          f"NaN={df_tracks_o[col].isna().sum():,}")


In [ ]:
# ── Variable lists for lifetime aggregation (user-adjustable) ─────────────────
# Each variable produces an output column named  {var}_lt_{op}

max_vars = [
    'ccs_area_all',   # Total CCS area including merged/split systems (km²)
    'ccs_area',       # Cold cloud shield area (km²)
    'core_area',      # Cold core area (km²)
    'pf_area',        # Precipitation feature area (km²)
    'pf_majoraxis',   # PF major axis length (km)
    'pf_maxrainrate', # PF maximum rain rate (mm h⁻¹)
    'pf_rainrate',    # PF mean rain rate (mm h⁻¹)
    'rainrate_p95',   # PF rain rate 95th percentile (mm h⁻¹)
]
min_vars = [
    'corecold_mintb', # Minimum cold core brightness temperature (K)
    'core_meantb',    # Mean convective core brightness temperature (K)
]
median_vars = ['movement_speed', ] # MCS movement speed (m s⁻¹)
mean_vars = [
    'heavy_rain_ratio', # Ratio of heavy rain area to total PF area (lifetime mean)
    'movement_speed',   # MCS movement speed (m s⁻¹)
    # 'pf_landfrac',    # PF land fraction (lifetime mean)
    # 'meanlat',        # Cloud centroid latitude (°)
    # 'meanlon',        # Cloud centroid longitude (°)
]
sum_vars = [
    'total_volrain',      # Total volumetric precipitation (mm h⁻¹ km²)
    'total_heavyvolrain', # Total volumetric heavy precipitation (mm h⁻¹ km²)
    'total_rain',         # Total precipitation (mm h⁻¹)
    'total_heavyrain',    # Total heavy precipitation (mm h⁻¹)
]

In [ ]:
def compute_lifetime_stats(df_ts, max_vars, min_vars, mean_vars, median_vars, sum_vars=None):
    """
    Reduce a time-step level DataFrame to one row per (source, local_track).

    Aggregations applied:
      {var}_lt_max    for variables in max_vars
      {var}_lt_min    for variables in min_vars
      {var}_lt_mean   for variables in mean_vars
      {var}_lt_median for variables in median_vars
      {var}_lt_sum    for variables in sum_vars

    Parameters
    ----------
    df_ts    : pd.DataFrame  — time-step level data (output of load_combined_sources)
    *_vars   : list of str   — column names to aggregate

    Returns
    -------
    pd.DataFrame with one row per track.
    """
    if sum_vars is None:
        sum_vars = []

    # Filter to only variables that actually exist in the DataFrame
    def _present(var_list):
        return [v for v in var_list if v in df_ts.columns]

    agg_dict = {
        'track_duration_h': ('track_duration_h', 'first'),
        'period':            ('period',            'first'),  # provenance: source year
        'surface':           ('surface',           'first'),  # Land / Ocean label
        **{f'{c}_lt_max':    (c, 'max')    for c in _present(max_vars)},
        **{f'{c}_lt_min':    (c, 'min')    for c in _present(min_vars)},
        **{f'{c}_lt_mean':   (c, 'mean')   for c in _present(mean_vars)},
        **{f'{c}_lt_median': (c, 'median') for c in _present(median_vars)},
        **{f'{c}_lt_sum':    (c, 'sum')    for c in _present(sum_vars)},
    }

    return (
        df_ts
        .groupby(['source', 'local_track'], sort=False)
        .agg(**agg_dict)
        .reset_index()
    )


# ── Run ───────────────────────────────────────────────────────────────────────
print('Computing lifetime stats for land tracks ...')
df_lt_l = compute_lifetime_stats(df_tracks_l, max_vars, min_vars, mean_vars, median_vars, sum_vars)
print(f'  df_lt_l: {df_lt_l.shape}')

print('Computing lifetime stats for ocean tracks ...')
df_lt_o = compute_lifetime_stats(df_tracks_o, max_vars, min_vars, mean_vars, median_vars, sum_vars)
print(f'  df_lt_o: {df_lt_o.shape}')

# Combine into a single per-track DataFrame with 'surface' and 'source' columns
# for the Step-7 grouped boxplot  (x='surface', hue='source').
df_lt_all = pd.concat([df_lt_l, df_lt_o], ignore_index=True)

# Sanity check: land and ocean track sets should be disjoint
land_set  = set(zip(df_lt_l['source'], df_lt_l['local_track']))
ocean_set = set(zip(df_lt_o['source'], df_lt_o['local_track']))
n_overlap = len(land_set & ocean_set)
print(f'\nOverlap between land and ocean track sets: {n_overlap} (should be 0)')
print(f'\nCombined df_lt_all: {df_lt_all.shape[0]:,} tracks')
print(df_lt_all.groupby(['source', 'surface']).size().rename('tracks').to_frame())
df_lt_l.head()


---
## Step 7 — Box / Violin plots of lifetime statistics

`plot_boxplots()` creates an M × N grid of panels, one variable per panel.

**Two plot types** (controlled by `plot_type`):
- `'violin'` — filled violin + narrow white box-whisker overlay + mean dot
- `'box'`    — colored box-whisker only (same whisker/mean settings)

**Whiskers** span user-defined percentiles (default 5–95%).
**Mean** shown as a white-filled circle.
**Median** shown as a horizontal black line (optionally notched).

In [ ]:
# ── Source order, surface order, and colors (user-adjustable) ─────────────────
SOURCE_ORDER  = ['IMERG', 'SCREAM']       # hue order; must match SOURCE_LABELS values
SURFACE_ORDER = ['Land', 'Ocean']         # x-axis group order
SOURCE_COLORS = {
    'IMERG':  '#6E6E6E',   # gray
    'SCREAM': 'dodgerblue',
    # add more sources here as needed
}


In [ ]:
def plot_boxplots(df_lt, variables, var_labels=None, plot_type='box',
                  source_order=None, source_colors=None, surface_order=None,
                  whis=(5, 95), showmean=True, notch=False, showcaps=True,
                  figsize=None, figname=None, xtick_rotation=0, box_width=0.55,
                  panel_titles=None, yscale=None, suptitle=None,
                  fontsize=12, hspace=0.4, wspace=0.35):
    """
    Plot M x N panels of grouped box plots comparing MCS lifetime statistics
    across surface types (Land / Ocean) and data sources.

    Each panel:
      - x-axis : surface type   (Land / Ocean)
      - hue    : data source    (IMERG / SCREAM / ..., extensible)

    Parameters
    ----------
    df_lt         : pd.DataFrame   -- per-track lifetime stats; must contain
                                      'surface' and 'source' columns
    variables     : list of lists  -- 2-D grid of column names; None -> empty panel
    var_labels    : dict           -- {col_name: y-axis label}.  Defaults to col_name.
    plot_type     : str            -- 'box' (default; kept for API compatibility)
    source_order  : list of str    -- hue category order  (default: SOURCE_ORDER)
    source_colors : dict           -- {source: color}     (default: SOURCE_COLORS)
    surface_order : list of str    -- x-axis order        (default: SURFACE_ORDER)
    whis          : (lo%, hi%)     -- whisker percentiles (default 5, 95)
    showmean      : bool           -- show mean as filled circle (default True)
    notch         : bool           -- notched box (default False)
    showcaps      : bool           -- show whisker caps (default True)
    box_width     : float          -- box width (default 0.55)
    figsize       : (w, h) in inches; auto-computed if None
    figname       : str or None    -- path to save figure
    panel_titles  : list of lists  -- custom panel title strings (optional)
    yscale        : list of lists  -- 'linear'/'log' per panel
    suptitle      : str or None    -- figure-level title
    fontsize      : int            -- base font size
    hspace/wspace : float          -- subplot spacing
    """
    if source_order  is None: source_order  = SOURCE_ORDER
    if source_colors is None: source_colors = SOURCE_COLORS
    if surface_order is None: surface_order = SURFACE_ORDER
    if var_labels    is None: var_labels    = {}

    nrows = len(variables)
    ncols = max(len(row) for row in variables)
    if figsize is None:
        figsize = (ncols * 4.5, nrows * 4)

    mpl.rcParams['font.size'] = fontsize
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)

    # Track counts per source (across all surfaces, for legend labels)
    track_counts = {src: int((df_lt['source'] == src).sum())
                    for src in source_order}

    meanpointprops = dict(
        marker='o', markersize=7,
        markerfacecolor='r', markeredgecolor='whitesmoke',
        markeredgewidth=1.5, zorder=4,
    )
    medianprops = dict(color='w', linewidth=2, zorder=5)

    legend_added = False   # add legend to the first visible panel only

    for ri, row_vars in enumerate(variables):
        for ci in range(ncols):
            ax = axes[ri][ci]
            var = row_vars[ci] if ci < len(row_vars) else None

            if var is None or var not in df_lt.columns:
                ax.set_visible(False)
                continue

            plot_data = df_lt[df_lt[var].notna()].copy()

            # ── Grouped box: x = surface type, hue = source ───────────────────
            sns.boxplot(
                data=plot_data,
                x='surface', y=var,
                hue='source',
                order=surface_order,
                hue_order=source_order,
                palette=source_colors,
                dodge=True,
                width=box_width,
                whis=list(whis),
                showfliers=False,
                notch=notch,
                medianprops=medianprops,
                whiskerprops=dict(color='black', linewidth=1.2),
                capprops=dict(color='black', linewidth=1.2),
                showcaps=showcaps,
                showmeans=showmean,
                meanprops=meanpointprops,
                ax=ax,
            )

            # ── Panel title ────────────────────────────────────────────────────
            if panel_titles and ri < len(panel_titles) and ci < len(panel_titles[ri]):
                ax.set_title(panel_titles[ri][ci], loc='left', fontweight='bold')

            # ── Cosmetics ──────────────────────────────────────────────────────
            scale = (yscale[ri][ci] if yscale and ri < len(yscale)
                     and ci < len(yscale[ri]) else 'linear')
            ax.set_yscale(scale)

            ax.set_ylabel(var_labels.get(var, var))
            ax.set_xlabel('')
            ax.grid(axis='y', ls='--', alpha=0.7)
            ax.tick_params(axis='both', labelsize=fontsize)
            ax.tick_params(axis='x', rotation=xtick_rotation)
            if xtick_rotation != 0:
                for lbl in ax.get_xticklabels():
                    lbl.set_horizontalalignment('right')

            for spine in ax.spines.values():
                spine.set_color('black')

            # ── Legend ─────────────────────────────────────────────────────────
            # Source names with total track counts; first visible panel only.
            if not legend_added:
                handles, labels = ax.get_legend_handles_labels()
                new_labels = [
                    f'{lbl}  (n={track_counts.get(lbl, 0):,})' for lbl in labels
                ]
                ax.legend(handles, new_labels,
                          title='', fontsize=fontsize - 1,
                          title_fontsize=fontsize - 1,
                          loc='best', framealpha=0.85)
                legend_added = True
            else:
                leg = ax.get_legend()
                if leg is not None:
                    leg.remove()

    fig.subplots_adjust(hspace=hspace, wspace=wspace)

    if suptitle is not None:
        fig.suptitle(suptitle, y=0.95, fontsize=fontsize * 1.5, fontweight='bold')

    if figname:
        fig.savefig(figname, dpi=150, bbox_inches='tight', facecolor='w')
        print(f'Saved: {figname}')
    return fig


In [ ]:
mpl.rcParams['font.family'] = 'STIXGeneral'
# mpl.rcParams['font.family'] = 'DejaVuSans'
# mpl.rcParams['font.family'] = 'Helvetica'

In [ ]:
# ── Variable grid, scales, and axis labels (user-adjustable) ─────────────────
lt_variables = [
    ['track_duration_h',         'ccs_area_all_lt_max',      'corecold_mintb_lt_min'],
    ['pf_majoraxis_lt_max',      'pf_rainrate_lt_max',       'rainrate_p95_lt_max'],
    ['movement_speed_lt_median', 'total_volrain_lt_sum',     'heavy_rain_ratio_lt_mean'],
]
panel_titles = [
    ['(a) Lifetime',              '(b) Max CCS Area',         '(c) Min Tb'],
    ['(d) Max PF Major Axis',     '(e) Max PF Mean Rain Rate','(f) Max P95 Rain Rate'],
    ['(g) Median Movement Speed', '(h) Total Volumetric Rain','(i) Mean Heavy Rain Ratio'],
]
lt_yscale = [
    ['linear', 'log',    'linear'],
    ['linear', 'linear', 'linear'],
    ['linear', 'log',    'linear'],
]
lt_labels = {
    'track_duration_h':           'Track Duration (h)',
    'ccs_area_all_lt_max':        'CCS Area (km²)',
    'ccs_area_lt_max':            'CCS Area (km²)',
    'pf_area_lt_max':             'PF Area (km²)',
    'pf_majoraxis_lt_max':        'Major Axis Length (km)',
    'pf_rainrate_lt_max':         'Mean Rain Rate (mm/h)',
    'rainrate_p95_lt_max':        'P95 Rain Rate (mm/h)',
    'movement_speed_lt_median':   'Movement Speed (m/s)',
    'corecold_mintb_lt_min':      'Min Tb (K)',
    'total_volrain_lt_sum':       'Rain Volume (mm km²)',
    'heavy_rain_ratio_lt_mean':   'Heavy Rain Ratio',
}

suptitle = 'Tropical MCS Lifetime Statistics — Land vs Ocean'
fig = plot_boxplots(
    df_lt_all, lt_variables,
    var_labels=lt_labels,
    plot_type='box',
    suptitle=suptitle,
    yscale=lt_yscale,
    xtick_rotation=0,
    box_width=0.4,
    whis=(5, 95), notch=True,
    panel_titles=panel_titles,
    fontsize=12,
    figsize=(13.5, 12),
    hspace=0.3, wspace=0.35,
    figname=f'{figdir_out}mcs_lifetime_stats_land_ocean_boxplot.png',
)
plt.show()


## Composite time-evolution of cell track properties

In [ ]:
def composite_track_evolution(
    df, var,
    duration_range,
    dataset_col='source',
    time_col='relative_step',
    duration_col='track_duration',
    track_col='local_track',
):
    """
    Compute composite time-evolution statistics for tracks within a duration range.

    Parameters
    ----------
    df : pd.DataFrame
        Tidy DataFrame with one row per (track, time-step).
    var : str
        Column name of the variable to composite.
    duration_range : tuple (lo, hi)
        Half-open interval [lo, hi) of track duration in hours.
    dataset_col, time_col, duration_col, track_col : str
        Column name overrides.

    Returns
    -------
    dict with keys:
        'time'          : 1-D array of relative time values (hours)
        'median'        : median  of var at each time step
        'q25'           : 25th percentile
        'q75'           : 75th percentile
        'mean'          : mean of var at each time step
        'n'             : number of tracks contributing at each time step
        'n_tracks'      : total qualifying tracks in this bin
        'duration_range': the input duration_range tuple
    """
    lo, hi = duration_range

    # One unique duration value per track (broadcast constant, so any row works)
    track_dur = (
        df[[dataset_col, track_col, duration_col]]
        .drop_duplicates(subset=[dataset_col, track_col])
    )
    valid_tracks = track_dur.loc[
        (track_dur[duration_col] >= lo) & (track_dur[duration_col] < hi),
        [dataset_col, track_col]
    ]

    sub = df.merge(valid_tracks, on=[dataset_col, track_col], how='inner')
    if sub.empty:
        return None

    n_tracks = sub[[dataset_col, track_col]].drop_duplicates().shape[0]

    grp = sub.groupby(time_col)[var]
    result = pd.DataFrame({
        'time':   grp.median().index,
        'median': grp.median().values,
        'q25':    grp.quantile(0.25).values,
        'q75':    grp.quantile(0.75).values,
        'mean':   grp.mean().values,
        'n':      grp.count().values,
    })

    return {
        'time':           result['time'].values,
        'median':         result['median'].values,
        'q25':            result['q25'].values,
        'q75':            result['q75'].values,
        'mean':           result['mean'].values,
        'n':              result['n'].values,
        'n_tracks':       n_tracks,
        'duration_range': duration_range,
    }


In [ ]:
df_tracks_l.keys()

In [ ]:
df_tracks_l['start_split_cloudnumber']

In [ ]:
# ── Filter to non-merge / non-split tracks ───────────────────────────────────
# In PyFLEXTRKR encoding:
#   start_split_cloudnumber == FillValue: natural initiation (not from a split)
#   end_merge_cloudnumber   == FillValue: natural dissipation (not merged into another track)
df_nosplit = df_tracks_l[
    (np.isnan(df_tracks_l['start_split_cloudnumber'])) &
    (np.isnan(df_tracks_l['end_merge_cloudnumber']))
].copy()

print("All tracks:")
print(df_tracks_l[['source', 'local_track']].drop_duplicates().groupby('source').size())
print("\nNon-merge/non-split tracks:")
print(df_nosplit[['source', 'local_track']].drop_duplicates().groupby('source').size())

# ── Variable specs ────────────────────────────────────────────────────────────
# Add new variables here; that is the only change required to composite & plot them.
#   var      : column name in df_nosplit
#   ylabel   : y-axis label for plot
#   filetag  : filename fragment
comp_var_specs = {
    'ccs_area': dict(ylabel='CCS Area (km²)',  filetag='CCS Area'),
    'corecold_mintb': dict(ylabel='Min Tb (K)', filetag='Min Tb'),
    'pf_area': dict(ylabel='PF Area (km²)', filetag='PF Area'),
    'pf_majoraxis': dict(ylabel='PF Major Axis Length (km)', filetag='PF Major Axis'),
    'rainrate_p95': dict(ylabel='P95 Rain Rate (mm/h)', filetag='P95 Rain Rate'),
    'total_volrain': dict(ylabel='Total Rain Volume (mm/h km²)', filetag='Total Rain Volume'),
}

# ── Duration bins ─────────────────────────────────────────────────────────────
# bin_step     = 4    # hours
# max_duration = 28   # hours
# dur_bins     = [(lo, lo + bin_step) for lo in range(0, max_duration, bin_step)]
dur_bins = [
    (4, 10), (10, 14), (14, 18), (18, 22), (22, 26), (26, 30),
]

# ── Compute composites for all variables ──────────────────────────────────────
# all_composites[var][ds_label][bin_label] = result dict
all_composites = {}
for var in comp_var_specs:
    all_composites[var] = {}
    for ds_label in SOURCE_ORDER:
        all_composites[var][ds_label] = {}
        df_ds = df_nosplit[df_nosplit['source'] == ds_label]
        for lo, hi in dur_bins:
            bin_label = f'{lo}–{hi} h'
            result = composite_track_evolution(df_ds, var=var, duration_range=(lo, hi))
            if result is not None:
                all_composites[var][ds_label][bin_label] = result
    print(f'{var}: computed')

### Function to plot composite evolution for each lifetime bin as line

In [ ]:
def plot_composite_evolution(
    composites,
    ds_order,
    nrow=1,
    ncol=None,
    figsize=(7, 5),
    dpi=150,
    hspace=None,
    wspace=None,
    sharey=True,
    cmap_name='plasma_r',
    fontsize=12,
    xlabel='Relative Time (h)',
    ylabel=None,
    titles=None,
    suptitle=None,
    suptitle_y=1.12,
    legend_ncols=2,
    legend_title='Track duration',
    figname=None,
):
    """
    Plot composite time-evolution lines for an M×N panel grid.

    Parameters
    ----------
    composites : dict
        Nested dict: composites[ds_label][bin_label] = result dict from
        composite_track_evolution(). Each result dict must contain:
        'time', 'mean', 'duration_range', 'n_tracks'.
    ds_order : list of str
        Dataset labels in row-major (left→right, top→bottom) order.
        Length must equal nrow * ncol.
    nrow, ncol : int
        Grid dimensions. ncol defaults to len(ds_order) // nrow.
    figsize : tuple
    dpi : int
    hspace, wspace : float or None
        GridSpec row/column spacing. None → tight_layout handles spacing.
    sharey : bool
        Share the y-axis within each row.
    cmap_name : str
        Matplotlib colormap name for duration-bin lines.
    fontsize : int
    xlabel : str or list of str
        X-axis label(s). A single str is broadcast to all panels; a list must
        have length nrow*ncol.
    ylabel : str or list of str or None
        Y-axis label(s). A single str is applied to the leftmost panel of each
        row; a list of length nrow is applied one per row's left panel.
    titles : list of str or None
        Panel titles. For nrow>1, only the top row is titled (row 0).
        Defaults to ds_label for each top-row panel.
    suptitle : str or None
        Figure-level super-title.
    suptitle_y : float
        Vertical position of suptitle in figure-fraction coordinates.
    legend_ncols : int
    legend_title : str
    figname : str or None
        If given, save the figure to this path (bbox_inches='tight').

    Returns
    -------
    fig : matplotlib.figure.Figure
    axes_grid : list[list[Axes]]  — shape (nrow, ncol)
    """
    if ncol is None:
        ncol = len(ds_order) // nrow

    n_panels = nrow * ncol
    if len(ds_order) != n_panels:
        raise ValueError(
            f'len(ds_order)={len(ds_order)} must equal nrow*ncol={n_panels}'
        )

    # ── colormap ─────────────────────────────────────────────────────────────
    bin_labels = list(next(iter(composites.values())).keys())
    n_bins     = len(bin_labels)
    cmap_obj   = plt.get_cmap(cmap_name, n_bins)
    colors     = [cmap_obj(i) for i in range(n_bins)]

    # ── normalise per-panel label arguments ──────────────────────────────────
    xlabels_list = [xlabel] * n_panels if isinstance(xlabel, str) else list(xlabel)

    if ylabel is None:
        ylabels_list = [None] * nrow
    elif isinstance(ylabel, str):
        ylabels_list = [ylabel] * nrow
    else:
        ylabels_list = list(ylabel)   # one entry per row

    titles_list = list(titles) if titles is not None else list(ds_order)

    # ── figure & GridSpec ────────────────────────────────────────────────────
    gs_kw = {}
    if hspace is not None:
        gs_kw['hspace'] = hspace
    if wspace is not None:
        gs_kw['wspace'] = wspace

    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs  = fig.add_gridspec(nrow, ncol, **gs_kw)

    # Build axes; share y within each row
    axes_grid = []
    for r in range(nrow):
        row_axes = []
        for c in range(ncol):
            share_with = row_axes[0] if (sharey and c > 0) else None
            ax = fig.add_subplot(gs[r, c], sharey=share_with)
            row_axes.append(ax)
        axes_grid.append(row_axes)

    # ── plot each panel ───────────────────────────────────────────────────────
    for idx, (ds_label, title) in enumerate(zip(ds_order, titles_list)):
        r, c = divmod(idx, ncol)
        ax   = axes_grid[r][c]

        for i, bin_label in enumerate(bin_labels):
            result = composites[ds_label].get(bin_label)
            if result is None:
                continue
            lo, hi = result['duration_range']
            lbl    = f'{lo}–{hi} h  (n={result["n_tracks"]:,})'
            ax.plot(result['time'], result['mean'],
                    color=colors[i], lw=1.8, label=lbl)

        # Title: only top row for multi-row grids (always for single row)
        if nrow == 1 or r == 0:
            ax.set_title(title, fontsize=fontsize, weight='bold', pad=6)
        ax.grid(ls=':', alpha=0.75)
        ax.tick_params(labelsize=fontsize - 1)

        # Y-label on leftmost column of each row only
        if c == 0 and ylabels_list[r]:
            ax.set_ylabel(ylabels_list[r], fontsize=fontsize)

        # Suppress shared tick labels on non-leftmost columns
        if sharey and c > 0:
            ax.tick_params(labelleft=False)

        # X-label: always for single-row; only bottom row for multi-row
        if nrow == 1 or r == nrow - 1:
            ax.set_xlabel(xlabels_list[idx], fontsize=fontsize)
        else:
            ax.set_xlabel('')

        # Change the color of all four axis spines (borders)
        for spine in ax.spines.values():
            spine.set_color('black')

        # Legend: always for single-row; only top row for multi-row
        if nrow == 1 or r == 0:
            ax.legend(ncols=legend_ncols, title=legend_title,
                      fontsize=fontsize - 2, title_fontsize=fontsize - 1,
                      loc='lower center', bbox_to_anchor=(0.5, 1.08),
                      framealpha=0.85)

    # ── layout & suptitle ────────────────────────────────────────────────────
    if hspace is None and wspace is None:
        fig.tight_layout()

    if suptitle:
        fig.suptitle(suptitle, fontsize=fontsize + 1, weight='bold', y=suptitle_y)

    # ── time arrows (drawn after layout so axes positions are finalised) ──────
    # Use the actual xlabel bounding box to set y; axes fraction for x so the
    # arrow always spans the full axis width regardless of hspace/wspace/margins.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    gap_display = fontsize * 0.3 * (fig.dpi / 72)   # small gap below xlabel
    for ax in axes_grid[-1]:
        xlabel_bbox = ax.xaxis.label.get_window_extent(renderer=renderer)
        ax_bbox     = ax.get_window_extent(renderer=renderer)
        # y in axes-fraction coords (can be negative → below the axis spine)
        arrow_y_axes = (xlabel_bbox.y0 - gap_display - ax_bbox.y0) / ax_bbox.height
        ax.annotate('', xy=(1.0, arrow_y_axes), xytext=(0.0, arrow_y_axes),
                    xycoords='axes fraction', annotation_clip=False,
                    arrowprops=dict(arrowstyle='->', facecolor='k', lw=3))

    # ── save ─────────────────────────────────────────────────────────────────
    if figname:
        fig.savefig(figname, bbox_inches='tight', dpi=dpi)
        print(f'Saved: {figname}')

    plt.show()
    return fig, axes_grid


In [ ]:
SOURCE_LABELS

In [ ]:
# ── Combined variable x dataset composite plot ────────────────────────────────
# Build a flat composites dict keyed by '{var}|{ds_label}' so the generic
# plot function can address each panel independently.
_ds_pair    = SOURCE_ORDER   # ['IMERG', 'SCREAM'] — extensible for more sources
var_list    = list(comp_var_specs.keys())

combined_composites = {}   # combined_composites['{var}|{ds}'][bin_label]
panel_order     = []       # row-major panel keys  (nrow * ncol)
panel_titles    = []       # per-panel title
ylabels_per_row = []       # one ylabel per row (leftmost column)

for var, spec in comp_var_specs.items():
    ylabels_per_row.append(spec['ylabel'])
    for ds_label in _ds_pair:
        key = f'{var}|{ds_label}'
        combined_composites[key] = all_composites[var][ds_label]
        panel_order.append(key)
        panel_titles.append(ds_label)

figname_all = f'{figdir_out}Composite_AllVars_byDuration_land_mcs.png'
figsize = (7 * len(_ds_pair), 3 * len(var_list))

fig, axes = plot_composite_evolution(
    combined_composites,
    ds_order       = panel_order,
    nrow           = len(var_list),
    ncol           = len(_ds_pair),
    figsize        = figsize,
    hspace         = 0.17,
    wspace         = 0.05,
    dpi            = 150,
    cmap_name      = 'plasma_r',
    fontsize       = 14,
    xlabel         = 'Relative Time (h)',
    ylabel         = ylabels_per_row,
    titles         = panel_titles,
    suptitle       = 'Composite Time-Evolution of Tropical MCS (Land)',
    suptitle_y     = 0.98,
    figname        = figname_all,
)
